In [1]:
import pandas as pd
import numpy as np
from collections import Counter


file_path = '/home/madhulika.gurazada/paper2/Data/Nino3.4_OG_SST1.xlsx'

nino34 = pd.read_excel(file_path)

nino34_1900_2024 = nino34[(nino34['Year'] >= 1900) & (nino34['Year'] <= 2024)]

def select_seasons(data):
    # Create DJF (December, January, February) within the function
    djf_data = pd.DataFrame(columns=['Year', 'Dec', 'Jan', 'Feb'])
    djf_data['Year'] = data['Year'] 
    djf_data['Dec'] = data['Dec']
    djf_data['Jan'] = data['Jan'].shift(-1)
    djf_data['Feb'] = data['Feb'].shift(-1)
    
    return djf_data

# Assuming you have already read the data into 'nino34_1900_2022'
djf_data = select_seasons(nino34_1900_2024)

# Define the columns for each season
seasonal_columns = {
    'DJF': ['Dec', 'Jan', 'Feb']
}

# Create a dictionary to hold the DataFrames
seasonal_data = {
    'DJF': djf_data
}

climatology_start_year = 1991
climatology_end_year = 2020

# Calculate seasonal means for each DataFrame
for season, columns in seasonal_columns.items():
    data = seasonal_data[season]
    data[f'{season}_Mean'] = data[columns].mean(axis=1)
    # Filter the data for the climatology years
    climatology_data = data[(data['Year'] >= climatology_start_year) & (data['Year'] <= climatology_end_year)]
    climatology_mean = climatology_data[f'{season}_Mean'].mean()
    climatology_std = climatology_data[f'{season}_Mean'].std()
    # Calculate Nino anomalies
    data[f'Nino_{season}_Anomaly'] = (data[f'{season}_Mean'] - climatology_mean)/climatology_std
    # Classify as El Niño, La Niña, or Neutral
    data[f'Nino_{season}_Classification'] = np.where(
    data[f'Nino_{season}_Anomaly'] >  1.0, 'Strong El Niño',
    np.where(
        data[f'Nino_{season}_Anomaly'] >  0.5, 'El Niño',
        np.where(
            data[f'Nino_{season}_Anomaly'] < -1.0, 'Strong La Niña',
            np.where(
                data[f'Nino_{season}_Anomaly'] < -0.5, 'La Niña',
                'Neutral'
            )
        )
    )
)

    
el_nino_years = {}
la_nina_years = {}
neutral_years = {}
str_el_nino_years = {}
str_la_nina_years = {}

for season in seasonal_columns.keys():
    data = seasonal_data[season]
    # Extract the years for each classification and filter for years between 1901 and 2024
    el_nino_years[season] = data[(data['Year'] >= 1901) & (data['Year'] <= 2024) & (data[f'Nino_{season}_Classification'] == 'El Niño')]['Year'].tolist()
    la_nina_years[season] = data[(data['Year'] >= 1901) & (data['Year'] <= 2024) & (data[f'Nino_{season}_Classification'] == 'La Niña')]['Year'].tolist()
    neutral_years[season] = data[(data['Year'] >= 1901) & (data['Year'] <= 2024) & (data[f'Nino_{season}_Classification'] == 'Neutral')]['Year'].tolist()
    str_el_nino_years[season] = data[(data['Year'] >= 1901) & (data['Year'] <= 2024) & (data[f'Nino_{season}_Classification'] == 'Strong El Niño')]['Year'].tolist()
    str_la_nina_years[season] = data[(data['Year'] >= 1901) & (data['Year'] <= 2024) & (data[f'Nino_{season}_Classification'] == 'Strong La Niña')]['Year'].tolist()

el_nino_years_1 = [year + 1 for year in el_nino_years['DJF']]
la_nina_years_1 = [year + 1 for year in la_nina_years['DJF']]
neutral_years_1 = [year + 1 for year in neutral_years['DJF']]
str_el_nino_years_1 = [year + 1 for year in str_el_nino_years['DJF']]
str_la_nina_years_1 = [year + 1 for year in str_la_nina_years['DJF']]

In [2]:
print(el_nino_years)
print(la_nina_years)
print(neutral_years)
print(str_el_nino_years)
print(str_la_nina_years)

el_nino_years = sorted(el_nino_years['DJF'] + str_el_nino_years['DJF'])
la_nina_years = sorted(la_nina_years['DJF'] + str_la_nina_years['DJF'])

el_nino_years   = {'DJF': el_nino_years}
la_nina_years   = {'DJF': la_nina_years}

{'DJF': [1904, 1905, 1913, 1914, 1923, 1939, 1941, 1963, 1968, 1976, 1977, 1987, 1994, 2002, 2006, 2014, 2018, 2019]}
{'DJF': [1903, 1908, 1909, 1910, 1917, 1922, 1924, 1933, 1938, 1950, 1954, 1964, 1967, 1971, 1983, 1984, 1985, 1995, 2000, 2005, 2008, 2011, 2017, 2020, 2021, 2022, 2024]}
{'DJF': [1901, 1906, 1907, 1912, 1915, 1919, 1920, 1921, 1926, 1927, 1928, 1929, 1931, 1932, 1934, 1935, 1936, 1937, 1943, 1944, 1945, 1946, 1947, 1948, 1951, 1952, 1953, 1956, 1958, 1959, 1960, 1961, 1962, 1966, 1969, 1974, 1978, 1979, 1980, 1981, 1989, 1990, 1992, 1993, 1996, 2001, 2003, 2004, 2012, 2013, 2016]}
{'DJF': [1902, 1911, 1918, 1925, 1930, 1940, 1957, 1965, 1972, 1982, 1986, 1991, 1997, 2009, 2015, 2023]}
{'DJF': [1916, 1942, 1949, 1955, 1970, 1973, 1975, 1988, 1998, 1999, 2007, 2010]}


In [3]:
print(el_nino_years)
print(la_nina_years)
print(neutral_years)
print(str_el_nino_years)
print(str_la_nina_years)

counts = {
    'El Niño':             len(el_nino_years   ['DJF']),
    'La Niña':             len(la_nina_years   ['DJF']),
    'Neutral':             len(neutral_years   ['DJF']),
    'Strong El Niño':      len(str_el_nino_years['DJF']),
    'Strong La Niña':      len(str_la_nina_years['DJF']),
}

print(counts)

{'DJF': [1902, 1904, 1905, 1911, 1913, 1914, 1918, 1923, 1925, 1930, 1939, 1940, 1941, 1957, 1963, 1965, 1968, 1972, 1976, 1977, 1982, 1986, 1987, 1991, 1994, 1997, 2002, 2006, 2009, 2014, 2015, 2018, 2019, 2023]}
{'DJF': [1903, 1908, 1909, 1910, 1916, 1917, 1922, 1924, 1933, 1938, 1942, 1949, 1950, 1954, 1955, 1964, 1967, 1970, 1971, 1973, 1975, 1983, 1984, 1985, 1988, 1995, 1998, 1999, 2000, 2005, 2007, 2008, 2010, 2011, 2017, 2020, 2021, 2022, 2024]}
{'DJF': [1901, 1906, 1907, 1912, 1915, 1919, 1920, 1921, 1926, 1927, 1928, 1929, 1931, 1932, 1934, 1935, 1936, 1937, 1943, 1944, 1945, 1946, 1947, 1948, 1951, 1952, 1953, 1956, 1958, 1959, 1960, 1961, 1962, 1966, 1969, 1974, 1978, 1979, 1980, 1981, 1989, 1990, 1992, 1993, 1996, 2001, 2003, 2004, 2012, 2013, 2016]}
{'DJF': [1902, 1911, 1918, 1925, 1930, 1940, 1957, 1965, 1972, 1982, 1986, 1991, 1997, 2009, 2015, 2023]}
{'DJF': [1916, 1942, 1949, 1955, 1970, 1973, 1975, 1988, 1998, 1999, 2007, 2010]}
{'El Niño': 34, 'La Niña': 39, 'Neutra

In [4]:
# 1) Build a year→phase mapping, giving strong phases priority
phase_by_year = {}
# Neutral
for y in neutral_years['DJF']:
    phase_by_year[y] = 'Neutral'   
# La Niña
for y in la_nina_years['DJF']:
    phase_by_year[y] = 'La Niña'
# El Niño
for y in el_nino_years['DJF']:
    phase_by_year[y] = 'El Niño'
# Override with strong La Niña
for y in str_la_nina_years['DJF']:
    phase_by_year[y] = 'Strong La Niña'
# Override with strong El Niño
for y in str_el_nino_years['DJF']:
    phase_by_year[y] = 'Strong El Niño'

In [5]:
phase_counts = Counter(phase_by_year.values())
print(phase_counts)

Counter({'Neutral': 51, 'La Niña': 27, 'El Niño': 18, 'Strong El Niño': 16, 'Strong La Niña': 12})


In [6]:
# 2) Sorted list of years
years = sorted(phase_by_year)

# 3) Count all year→year+1 transitions
trans = Counter()
for y in years:
    nxt = y + 1
    if nxt in phase_by_year:
        frm = phase_by_year[y]
        to  = phase_by_year[nxt]
        trans[(frm, to)] += 1

# 4) Print the requested transitions
print("Neutral → El Niño:",             trans[('Neutral',        'El Niño')])
print("El Niño → Neutral:",             trans[('El Niño',        'Neutral')])

print("Neutral → La Niña:",             trans[('Neutral',        'La Niña')])
print("La Niña → Neutral:",             trans[('La Niña',        'Neutral')])

print("Neutral → Strong El Niño:",      trans[('Neutral',        'Strong El Niño')])
print("Strong El Niño → Neutral:",      trans[('Strong El Niño','Neutral')])

print("Neutral → Strong La Niña:",      trans[('Neutral',        'Strong La Niña')])
print("Strong La Niña → Neutral:",      trans[('Strong La Niña','Neutral')])

print("La Niña → El Niño:",             trans[('La Niña','El Niño')])
print("El Niño → La Niña:",             trans[('El Niño','La Niña')])

print("Strong El Niño → La Niña:",      trans[('Strong El Niño','La Niña')])
print("La Niña → Strong El Niño:",      trans[('La Niña','Strong El Niño')])

print("Strong La Niña → El Niño:",      trans[('Strong La Niña','El Niño')])
print("El Niño → Strong La Niña:",      trans[('El Niño','Strong La Niña')])

print("Strong El Niño → Strong La Niña:", trans[('Strong El Niño','Strong La Niña')])
print("Strong La Niña → Strong El Niño:", trans[('Strong La Niña','Strong El Niño')])

Neutral → El Niño: 5
El Niño → Neutral: 5
Neutral → La Niña: 8
La Niña → Neutral: 5
Neutral → Strong El Niño: 6
Strong El Niño → Neutral: 8
Neutral → Strong La Niña: 4
Strong La Niña → Neutral: 4
La Niña → El Niño: 6
El Niño → La Niña: 4
Strong El Niño → La Niña: 3
La Niña → Strong El Niño: 8
Strong La Niña → El Niño: 1
El Niño → Strong La Niña: 3
Strong El Niño → Strong La Niña: 3
Strong La Niña → Strong El Niño: 0


In [7]:
# define your phase groups
la_phases = ['La Niña', 'Strong La Niña']
el_phases = ['El Niño', 'Strong El Niño']

# compute each aggregated transition
la_to_el     = sum(trans[(la, el)] for la in la_phases for el in el_phases)
el_to_la     = sum(trans[(el, la)] for el in el_phases for la in la_phases)
seln_to_la   = sum(trans[('Strong El Niño', la)] for la in la_phases)
la_to_seln   = sum(trans[(la, 'Strong El Niño')] for la in la_phases)
seln_to_slnn = trans[('Strong El Niño','Strong La Niña')]
slnn_to_seln = trans[('Strong La Niña','Strong El Niño')]

print("La Niña → El Niño:",             la_to_el)     # 15
print("El Niño → La Niña:",             el_to_la)     # 13
print("Strong El Niño → La Niña:",      seln_to_la)   # 6
print("La Niña → Strong El Niño:",      la_to_seln)   # 8
print("Strong El Niño → Strong La Niña:", seln_to_slnn) # 3
print("Strong La Niña → Strong El Niño:", slnn_to_seln) # 0

La Niña → El Niño: 15
El Niño → La Niña: 13
Strong El Niño → La Niña: 6
La Niña → Strong El Niño: 8
Strong El Niño → Strong La Niña: 3
Strong La Niña → Strong El Niño: 0


In [8]:
# your existing data
years      = sorted(phase_by_year.keys())


# helper to pull next‐year phase safely
def next_phase(y):
    return phase_by_year.get(y+1, None)

# build lists of the transition years (as tuples)
la_to_el_years   = [(y, y+1) for y in years
                    if phase_by_year[y] in la_phases and next_phase(y) in el_phases]

el_to_la_years   = [(y, y+1) for y in years
                    if phase_by_year[y] in el_phases and next_phase(y) in la_phases]

seln_to_la_years = [(y, y+1) for y in years
                    if phase_by_year[y]=='Strong El Niño' and next_phase(y) in la_phases]

la_to_seln_years = [(y, y+1) for y in years
                    if phase_by_year[y] in la_phases and next_phase(y)=='Strong El Niño']

seln_to_slnn_years = [(y, y+1) for y in years
                      if phase_by_year[y]=='Strong El Niño' and next_phase(y)=='Strong La Niña']

slnn_to_seln_years = [(y, y+1) for y in years
                      if phase_by_year[y]=='Strong La Niña' and next_phase(y)=='Strong El Niño']

# print them
print("La Niña → El Niño:",             la_to_el_years)
print("El Niño → La Niña:",             el_to_la_years)
print("Strong El Niño → La Niña:",      seln_to_la_years)
print("La Niña → Strong El Niño:",      la_to_seln_years)
print("Strong El Niño → Strong La Niña:", seln_to_slnn_years)
print("Strong La Niña → Strong El Niño:", slnn_to_seln_years)

La Niña → El Niño: [(1903, 1904), (1910, 1911), (1917, 1918), (1922, 1923), (1924, 1925), (1938, 1939), (1964, 1965), (1967, 1968), (1971, 1972), (1975, 1976), (1985, 1986), (2005, 2006), (2008, 2009), (2017, 2018), (2022, 2023)]
El Niño → La Niña: [(1902, 1903), (1923, 1924), (1941, 1942), (1963, 1964), (1972, 1973), (1982, 1983), (1987, 1988), (1994, 1995), (1997, 1998), (2006, 2007), (2009, 2010), (2019, 2020), (2023, 2024)]
Strong El Niño → La Niña: [(1902, 1903), (1972, 1973), (1982, 1983), (1997, 1998), (2009, 2010), (2023, 2024)]
La Niña → Strong El Niño: [(1910, 1911), (1917, 1918), (1924, 1925), (1964, 1965), (1971, 1972), (1985, 1986), (2008, 2009), (2022, 2023)]
Strong El Niño → Strong La Niña: [(1972, 1973), (1997, 1998), (2009, 2010)]
Strong La Niña → Strong El Niño: []
